In [ ]:
### Display entire dataset on a sphere
import pandas as pd
import plotly.graph_objects as go
import numpy as np

# Get continents data:
df_continents = pd.read_csv('lat_long_elev.csv')

# Add LEDs data:
strips_num = 10  # Number of LED strips
LEDs_per_strip = 40  # Number of LEDs per strip

longitudes = []
latitudes = []
elevations = []

# Generate dicrete LEDS positions
for strip in range(strips_num):
    long = -180 + 360/strips_num*strip
    for led in range(LEDs_per_strip):
        lat = -90 + 180/LEDs_per_strip*led

        longitudes.append(long)
        latitudes.append(lat)
        elevations.append(100000)
df_LEDs = pd.DataFrame({'lat': latitudes, 'long': longitudes, 'elev': elevations})

def computeDisplayDf(continents_df, LEDs_df):
    df = pd.concat([continents_df, LEDs_df], ignore_index=True)

    # Earth radius in kilometers (approximate)
    earth_radius = 6371

    # Convert lat/long/elevation to 3D Cartesian coordinates on a sphere
    lat_rad = np.radians(df['lat'])
    long_rad = np.radians(df['long'])
    radius = earth_radius + df['elev']/1000

    x = radius * np.cos(lat_rad) * np.cos(long_rad)
    y = radius * np.cos(lat_rad) * np.sin(long_rad)
    z = radius * np.sin(lat_rad)

    return pd.DataFrame({'x': x, 'y': y, 'z': z, 'lat': df['lat'], 'long': df['long'], 'elev': df['elev']})

def display(df):
    # Create spherical 3D scatter plot
    fig_sphere = go.Figure(data=[go.Scatter3d(
        x=df['x'],
        y=df['y'],
        z=df['z'],
        mode='markers',
        marker=dict(
            size=3,
            color=df['elev'],  # Color points by elevation
            colorscale='Viridis',
        ),
        text=[f'Lat: {lat:.1f}°<br>Long: {long:.1f}°<br>Elev: {elev:.1f}m' 
            for lat, long, elev in zip(df['lat'], df['long'], df['elev'])],
        hoverinfo='text'
    )])

    # Update layout
    fig_sphere.update_layout(
        scene=dict(
            xaxis_title='X (km)',
            yaxis_title='Y (km)',
            zaxis_title='Z (km)',
            aspectmode='data'
        ),
        width=900,
        height=900
    )

    return fig_sphere

df = computeDisplayDf(df_continents, df_LEDs)
display(df)

In [ ]:
### Get sun altitude
import suncalc
import datetime
import dash
from dash import dcc, html
from dash.dependencies import Output, Input

date = datetime.datetime.now() + datetime.timedelta(days=150)

# Get sun elevation angles
def updateLEDSToDate(df_LEDs,date):
    elevationAngles = pd.DataFrame(suncalc.get_position(date, df_LEDs['long'], df_LEDs['lat']))['altitude']*100000
    # Update LEDs elevation angles (values)
    df_LEDs['elev'] = elevationAngles
    df_LEDs['elev'] = df_LEDs['elev'].clip(lower=0)

# Initialize Dash app
app = dash.Dash(__name__)

updateLEDSToDate(df_LEDs, date)
df = computeDisplayDf(df_continents, df_LEDs)
fig = display(df)

# App layout
app.layout = html.Div([
    html.H3("Dynamic 3D Scatter Plot (Plotly + Dash)"),
    dcc.Graph(id='live-3d-scatter', figure=fig),
    dcc.Interval(
        id='interval-component',
        interval=100,  # Update every 1 seconds
        n_intervals=-1
    )
])

# Callback to update the figure
@app.callback(
    Output('live-3d-scatter', 'figure'),
    Input('interval-component', 'n_intervals')
)
def update_graph(n):
    date = datetime.datetime.now()+datetime.timedelta(days=1,hours=2*n)
    updateLEDSToDate(df_LEDs, date)
    df = computeDisplayDf(df_continents, df_LEDs)
    fig = display(df)
    print(date)
    return fig

# Run the app
app.run(debug=True)